In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')

Aim - build a preprocessing pipeline and a baseline classification model. 

Logistic Regression is used, with L2 regularisation and class_weights='balanced' to deal with the correlated features and imbalance in target labels.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(transform_output='pandas')

# pop target labels
y = df.pop('health_condition')

# custom functions for preprocessing
def add_na_count(X):
    X = X.copy()
    X['na_count'] = X.isna().sum(axis=1)
    return X

def add_prod(X):
    X = X.copy()
    X['activity_x_duration'] = (
        X['ordinal_encoding__physical_activity_level']
        * X['numerical__exercise_duration']
    )
    return X

def drop_id(X):
    X = X.copy()
    return X.drop(columns="id")

# separate pipelines for numerical, ordinal, and nominal data
nom_pre = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='other')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cats = [
    ['non-veg', 'balanced', 'veg'],
    ['low', 'medium', 'high'],
    ['poor', 'average', 'good'],
    ['sedentary', 'moderate', 'active'],
    ['no', 'occasional', 'yes']
]

ord_pre = Pipeline([
    ('encode', OrdinalEncoder(categories=cats, handle_unknown='use_encoded_value',
        unknown_value=np.nan, encoded_missing_value=np.nan)),
    ('impute', SimpleImputer(strategy='median'))
])

num_pre = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])

ordinal = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol']
numeric = ['sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake']

# combine preprocessing steps into one Column Transformer
encode_impute = ColumnTransformer([
    ('gender_encoding', nom_pre, ['gender']),
    ('ordinal_encoding', ord_pre, ordinal),
    ('numerical', num_pre, numeric)
], remainder='passthrough')

# define classification model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
#   l1_ratio=0, not used in this version of sklearn
    C=0.1, 
    class_weight='balanced',
    max_iter=300
)

# create final classification pipeline
pipe = Pipeline([
    ('drop_id', FunctionTransformer(drop_id, validate=False)),
    ('na_counter', FunctionTransformer(add_na_count, validate=False)),
    ('encode_and_impute', encode_impute),
    ('add_product', FunctionTransformer(add_prod, validate=False)),
    ('scaler', StandardScaler()), # for efficiency with Logistic Regression solver
    ('classifier', model)
])
pipe

Pipeline(steps=[('drop_id',
                 FunctionTransformer(func=<function drop_id at 0x7a26173b6de0>)),
                ('na_counter',
                 FunctionTransformer(func=<function add_na_count at 0x7a262c1bf740>)),
                ('encode_and_impute',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('gender_encoding',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='other',
                                                                                 strategy='constant')),
                                                                  ('enc...
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['sleep_duration',
                                                   'heart_rate', 'bmi',
                                                   'calorie_expenditure',
                                                   'step_count',
                                                   'exercise_duration',
                                                   'water_intake'])])),
                ('add_product',
                 FunctionTransformer(func=<function add_prod at 0x7a26173b6c00>)),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=300))])

In [3]:
# split into stratified folds, cross validate and evaluate balanced accuracy score
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

scores = cross_val_score(
    pipe,
    df,
    y,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1
)

# to compare to other models
print(scores)
print(f'Baseline score (Logistic Regression): {scores.mean()}')

[0.87851415 0.87901586 0.87848997 0.88091346 0.87875299]
Baseline score (Logistic Regression): 0.8791372843994019
